# 앙상블 검색기(Ensemle Retriever)

## 특징
 - 1. 여러 검색기 통합:다양한 유형의 검색기를 입력으로 받아 결과를 재결합
 - 2. 결과 재순위: Reciprocal Rank Fusion 알고리즘을 사용하여 결과의 순위를 조정
 - 3. 하이브리드 검색: 주로 sparse retriever와 dense retriever(예: 임베딩 유사도)를 결합하여 사용

## 장점
 - Sparse retriever: 키워드 기반 검색에 효과적이다.
 - Dense retriever: 의미적 유사성 기반 검색에 효과


In [ ]:
!pip install -q langchain pypdf sentence-transformers chromadb openai

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface chromadb

In [ ]:
!pip install langchain==0.1.16

In [ ]:
!pip install faiss-gpu-cu12

In [1]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

In [2]:
from langchain.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sbert-nli"
encode_kwargs = {'normalize_embeddings': True}
embedding = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs = encode_kwargs
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!pip install rank_bm25

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader

# 샘플 문서 리스트들
doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple's macbook",
]

# bm25 retriever와 faiss retriever를 초기화
bm25_retriever = BM25Retriever.from_texts(
    doc_list,
)
bm25_retriever.k = 1 #BM25Retriever의 검색 결과 개수를 1로 설정.
faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding,
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k":1})

# 앙상블 retriever를 초기화합니다.
ensemble_retriever = EnsembleRetriever(
    retrievers = [bm25_retriever, faiss_retriever],
    weights=[0.7,0.3]
)

In [8]:
# 검색 결과 문서를 가져옴
query ="my best favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력합니다.
print("[Ensemble Retriever]")
for doc in ensemble_result:
  print(f"Content: {doc.page_content}")
  print()

print("[BM25 Retriever]")
for doc in bm25_result:
  print(f"Content: {doc.page_content}")
  print()

print("[FAISS Retriever]")
for doc in faiss_result:
  print(f"Content: {doc.page_content}")
  print()

[Ensemble Retriever]
Content: Apple is my favorite company

Content: I like apples

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [9]:
# 검색 결과 문서를 가져옵니다.
query = "Apple company makes my favorite iphone"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result = faiss_retriever.invoke(query)

# 가져온 문서를 출력합니다.
print("[Ensemble Retrieval]")
for doc in ensemble_result:
  print(f"Content: {doc.page_content}")
  print()

print("[BM25 Retriever]")
for doc in bm25_result:
  print(f"Content: {doc.page_content}")
  print()

print("[FAISS Retriever]")
for doc in faiss_result:
  print(f"Content: {doc.page_content}")
  print()

[Ensemble Retrieval]
Content: Apple is my favorite company

Content: I like apple's iphone

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apple's iphone



In [10]:
from langchain_core.runnables import ConfigurableField

ensemble_retriever = EnsembleRetriever(
    # 리트리버 목록을 설정. 여기서는 bm25_retriever와 faiss_retriever를 사용.
    retrievers=[bm25_retriever, faiss_retriever],
).configurable_fields(
    weights=ConfigurableField(
        # 검색 매개변수의 고유 식별자를 설정합니다.
        id = "ensemble_weights",
        # 검색 매개변수의 이름을 설정합니다.
        name = "Ensemble Weights",
        # 검색 매개변수에 대한 설명을 작성합니다.
        description="Ensemble Weights"
    )
)

In [12]:
config = {"configurable": {"ensemble_weights": [1,0]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs # 검색 결과인 docs를 출력합니다.

[Document(page_content='Apple is my favorite company'),
 Document(page_content='I like apples')]

In [13]:
config = {"configurable": {"ensemble_weights": [0,1]}}

# config 매개변수를 사용하여 검색 설정을 지정합니다.
docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs # 검색 결과인 docs를 출력합니다.

[Document(page_content='I like apples'),
 Document(page_content='Apple is my favorite company')]